# 04. 3개 지수 × 모델 크기 백테스트 + 리포트
모델 실행 → 평가 → 그림을 이 노트북 안에서 끝냅니다. 결과는 변수(`fc`, `groups`, `tau`)에만 있고 파일로 저장하지 않습니다 (커널을 닫으면 사라지니 오래 걸리는 실행 뒤에는 필요한 그림부터 복사해 두세요).

**핵심 질문** ① 크기가 커지면 꼬리(q0.05) 캘리브레이션이 좋아지는가? ② ARIMA 대비 위기 직전·직후에 더 잘 반응하는가?

실행 시간 팁: 처음에는 `TICKERS=["KOSPI"]`, `MODELS=["timestral-5M", "arima", "climatology"]`, `ORIGIN_STEP=20` 으로 돌려 확인하세요.

In [1]:
import sys, pathlib, warnings
cwd = pathlib.Path.cwd()          # 커널 cwd 가 notebooks/ 든 프로젝트 루트든 상관없이 루트를 찾는다
ROOT = next(p for p in [cwd, *cwd.parents] if (p / "configs").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 100

In [2]:
TICKERS = ["SP500", "NASDAQ", "KOSPI"]
MODELS = ["timestral-5M", "timestral-28M", "timestral-123M", "timestral-209M", "arima", "climatology"]
CTXS = [512, 1024]                  # 설계안은 [256, 512, 1024, 2048]
ORIGIN_STEP = 5
EXPERIMENTS = None                  # None: configs/targets.yaml 의 4개 실험 전부
FAN_MODEL, FAN_CTX = "timestral-209M", 1024

In [ ]:
from src.eval.backtest import run_forecasts
from src.eval.summarize import summarize
from src.eval.bands import classify_mdd
from src.viz.plots import select, fan_chart, calibration_plot, size_comparison

fc = run_forecasts(tickers=TICKERS, models=MODELS, experiments=EXPERIMENTS, ctxs=CTXS, origin_step=ORIGIN_STEP)
print(fc.shape)
groups, tau = summarize(fc, origin_step=ORIGIN_STEP)
EXPS = fc[["series", "horizon"]].drop_duplicates().itertuples(index=False, name=None)
EXPS = list(EXPS)
EXPS

c:\vscode\Timestral_Experiment_in_Finance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
forecast runs:  79%|███████▊  | 165/210 [9:12:04<11:30:12, 920.29s/it] 

## 1. 전체 성능 요약 (regime = all)
`n_eff` 는 겹치지 않는 윈도우 기준 유효 표본 수 — 이 숫자가 작으면 순위 차이에 큰 의미를 두지 마세요.

In [ ]:
cols = ["model", "ticker", "ctx", "n", "n_eff", "mase_median", "avg_pinball", "coverage80", "ms_per_forecast"]
for series, h in EXPS:
    print(f"=== {series} h{h} ===")
    display(groups.query("regime == 'all' and series == @series and horizon == @h").sort_values(["ticker", "avg_pinball"])[cols].round(4))

## 2. 꼬리 캘리브레이션 — 모델 크기별 위반율 / pinball
위반율 점선 = 목표(tau). MDD 는 하위 꼬리(0.05, 0.10), 변동성은 상단 꼬리(0.90)도 확인.

In [ ]:
for series, h in EXPS:
    taus_ = [0.90] if series.startswith("yz") else [0.05, 0.95] if series.startswith("r") else [0.05, 0.10]
    for t in taus_:
        fig = size_comparison(tau, series, h, t)
        plt.show()

In [ ]:
# 꼬리 위반 검정표 (MDD, tau = 0.05)
tau.query("series in ['m5', 'm20'] and tau == 0.05 and regime == 'all'")[
    ["model", "ticker", "series", "horizon", "ctx", "hits", "violation_rate", "pinball", "kupiec_p", "christoffersen_p"]
].sort_values(["series", "ticker", "pinball"]).round(4)

In [ ]:
for series, h in EXPS:
    for tk in TICKERS:
        fig = calibration_plot(tau, tk, series, h, ctx=FAN_CTX)
        plt.show()

## 3. 위기 vs 평시
origin 날짜가 2008-09~2009-06, 2020-02~06, 2022 에 속하면 crisis.

In [ ]:
for series in ["m5", "m20", "r5", "r20"]:
    if series in set(fc.series):
        print(f"=== {series}: tau=0.05 위반율 (목표 0.05) ===")
        display(tau.query("series == @series and tau == 0.05 and regime != 'all'")
                .pivot_table(index=["ticker", "model", "ctx"], columns="regime", values="violation_rate").round(3))

## 4. Fan chart (대표 모델)

In [ ]:
for series, h in EXPS:
    for tk in TICKERS:
        d = select(fc, model=FAN_MODEL, ticker=tk, series=series, horizon=h, ctx=FAN_CTX)
        if len(d):
            fig = fan_chart(d, f"{FAN_MODEL} c{FAN_CTX} | {tk} {series} h{h}", show_upper=series.startswith("r"))
            plt.show()

## 5. 밴드 해석: 실현 MDD 가 사전 예측 밴드의 어디에 있었나
`panic(<q05)` 비율이 5% 근처여야 캘리브레이션이 맞는 것.

In [ ]:
x = select(fc, model=FAN_MODEL, series="m5", ctx=FAN_CTX)
if len(x):
    x = x.assign(state=classify_mdd(x))
    display(x.groupby("ticker")["state"].value_counts(normalize=True).unstack().round(3))

## 5-2. 수익률 분포: 양측 꼬리 캘리브레이션
수익률은 q0.05(하방)와 q0.95(상방)를 함께 봅니다. 위반율이 각각 0.05 / 0.95 에 가깝고 Kupiec p 가 작지 않아야 분포가 맞는 것. `r1`(1일)·`r5`·`r20` 은 앞으로 h일 구간의 누적 로그수익률.

In [ ]:
rt = tau.query("series in ['r1', 'r5', 'r20'] and tau in [0.05, 0.95] and regime == 'all'")
if len(rt):
    display(rt.pivot_table(index=["series", "ticker", "model", "ctx"], columns="tau", values=["violation_rate", "kupiec_p"]).round(3))

In [ ]:
# 실현 수익률이 사전 예측 분포의 어디에 있었나 (이상적: 하방 5% / 정상 90% / 상방 5%)
from src.eval.bands import classify_return
x = select(fc, model=FAN_MODEL, series="r5", ctx=FAN_CTX)
if len(x):
    x = x.assign(state=classify_return(x))
    display(x.groupby("ticker")["state"].value_counts(normalize=True).unstack().round(3))

## 6. 추론 시간 (이번 실행 기준, ms/forecast)

In [ ]:
groups.query("regime == 'all'").groupby(["model", "ctx"])["ms_per_forecast"].median().unstack().round(1)

In [ ]:
# (선택) 합성 데이터로 모델 크기별 순수 추론 지연시간만 따로 측정
from src.forecast.latency import measure_latency
from src.viz.plots import latency_bar
lat = measure_latency(sizes=["5M", "28M", "123M", "209M"], ctx=1024, horizon=20, n=64)
display(lat)
fig = latency_bar(lat)